In [18]:
# add directory root to path so we can import Pipit
import sys
sys.path.append("../../")

import pipit as pp

# path to sample trace file
dirname = "../../pipit/tests/data/ping-pong-hpctoolkit"

# load in trace file
trace = pp.Trace.from_hpctoolkit(dirname)

# Flat Profile
By default, calling `flat_profile()` generates a Pandas DataFrame that averages exclusive time spent in each function during program execution.

In [19]:
trace.flat_profile()

/home/nk/PSSG/pipit/docs/examples/../../pipit/trace.py:284: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  self.events.loc[enter_leave_mask] = enter_leave_df.groupby(
/home/nk/PSSG/pipit/docs/examples/../../pipit/trace.py:286: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  ).apply(_match_caller_callee_by_level)


,Name,Time (%),Avg Time (ns),Avg Calls,Min (ns),Max (ns)
0,__GI_process_vm_readv [libc-2.17.so],49.14,61892500.0,7.5,52841000.0,70944000.0
1,psm2_mq_ipeek2 [libpsm2.so.2.2],21.82,27479500.0,8.5,24389000.0,30570000.0
2,psm_progress_wait [libmpi.so.12.1.1],17.16,21609000.0,10.5,12651000.0,30567000.0
3,targ5030 [libpsm2.so.2.2],7.19,9055500.0,26.0,0.0,18111000.0
4,<unknown procedure> 0x24680 [libpsm2.so.2.2],4.69,5910500.0,6.0,0.0,11821000.0
5,__GI___munmap [libc-2.17.so],0.01,12000.0,1.0,12000.0,12000.0
6,MPI_Finalize,0.00,0.0,1.0,0.0,0.0
7,MPID_Recv [libmpi.so.12.1.1],0.00,0.0,7.0,0.0,0.0
8,MPID_Finalize [libmpi.so.12.1.1],0.00,0.0,1.0,0.0,0.0
9,PMPI_Finalize [libmpi.so.12.1.1],0.00,0.0,1.0,0.0,0.0


If we find that our flat profile consists of too many zero or NA values, we can simply pass in `drop_zeros=True`, which removes all rows where exclusive time is missing or 0.

In [20]:
trace.flat_profile(drop_zeros=True)

,Name,Time (%),Avg Time (ns),Avg Calls,Min (ns),Max (ns)
0,__GI_process_vm_readv [libc-2.17.so],49.14,61892500.0,7.5,52841000.0,70944000.0
1,psm2_mq_ipeek2 [libpsm2.so.2.2],21.82,27479500.0,8.5,24389000.0,30570000.0
2,psm_progress_wait [libmpi.so.12.1.1],17.16,21609000.0,10.5,12651000.0,30567000.0
3,targ5030 [libpsm2.so.2.2],7.19,9055500.0,26.0,0.0,18111000.0
4,<unknown procedure> 0x24680 [libpsm2.so.2.2],4.69,5910500.0,6.0,0.0,11821000.0
5,__GI___munmap [libc-2.17.so],0.01,12000.0,1.0,12000.0,12000.0


Looks much better! To further examine our trace, we can set `per_process=True` for a deeper look into where we're spending time.

In [21]:
trace.flat_profile(per_process=True, drop_zeros=True)

Time (ns)  Time (%)  \
Name                                         Process                         
__GI_process_vm_readv [libc-2.17.so]         0        70944000.0     57.31   
                                             1        52841000.0     42.69   
psm2_mq_ipeek2 [libpsm2.so.2.2]              0        24389000.0     44.38   
                                             1        30570000.0     55.62   
psm_progress_wait [libmpi.so.12.1.1]         1        12651000.0     29.27   
                                             0        30567000.0     70.73   
targ5030 [libpsm2.so.2.2]                    1        18111000.0    100.00   
<unknown procedure> 0x24680 [libpsm2.so.2.2] 1        11821000.0    100.00   
__GI___munmap [libc-2.17.so]                 0           12000.0    100.00   

                                                      Calls  
Name                                         Process         
__GI_process_vm_readv [libc-2.17.so]         0            8  
                                             1            7  
psm2_mq_ipeek2 [libpsm2.so.2.2]              0           11  
                                             1            6  
psm_progress_wait [libmpi.so.12.1.1]         1            7  
                                             0           14  
targ5030 [libpsm2.so.2.2]                    1           19  
<unknown procedure> 0x24680 [libpsm2.so.2.2] 1            5  
__GI___munmap [libc-2.17.so]                 0            1

We can also control which columns we'd like to group by, columns we'd like to have metrics of, and how our resulting DataFrame is ordered:

In [38]:
trace.flat_profile(metrics=["time.exc", "time.inc"], groupby_cols=["Name", "Calling Context ID"], per_process=True, drop_zeros=True, ascending=True)

,,,Time (ns),Time (%),Calls,time.inc (avg)
Name,Calling Context ID,Process,,,,
__GI___munmap [libc-2.17.so],157,0,12000.0,100.0,1,1.200000e+04
__GI_process_vm_readv [libc-2.17.so],98,0,5899000.0,100.0,1,5.899000e+06
<unknown procedure> 0x24680 [libpsm2.so.2.2],29,1,11821000.0,100.0,5,5.986400e+06
psm_progress_wait [libmpi.so.12.1.1],40,1,12651000.0,100.0,7,1.045043e+07
targ5030 [libpsm2.so.2.2],21,1,18111000.0,100.0,3,6.037000e+06
psm2_mq_ipeek2 [libpsm2.so.2.2],84,0,24389000.0,100.0,4,6.097250e+06
psm_progress_wait [libmpi.so.12.1.1],88,0,30567000.0,100.0,7,7.850857e+06
psm2_mq_ipeek2 [libpsm2.so.2.2],36,1,30570000.0,100.0,6,1.008367e+07
__GI_process_vm_readv [libc-2.17.so],50,1,52841000.0,100.0,7,7.548714e+06
